# ComfyUI Colab (r3)

r2からの追加改善点:
- r2の全改善を継承
- ✅ ControlNet Pose 対応セルを追加（DWPose / OpenPose）
- ✅ ControlNet モデル自動ダウンロード
- ✅ comfyui_controlnet_aux カスタムノード自動インストール


## Step 1: Environment Setup
ComfyUI のセットアップ、依存パッケージのインストールを行います。


In [ ]:
#@title Environment Setup

from pathlib import Path

OPTIONS = {}

USE_GOOGLE_DRIVE = False  #@param {type:"boolean"} Falseにすると100GBエリアへのインストールに切り替わります
UPDATE_COMFY_UI = True   #@param {type:"boolean"}
USE_COMFYUI_MANAGER = True  #@param {type:"boolean"}
INSTALL_CUSTOM_NODES_DEPENDENCIES = True  #@param {type:"boolean"}

OPTIONS['USE_GOOGLE_DRIVE'] = USE_GOOGLE_DRIVE
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI
OPTIONS['USE_COMFYUI_MANAGER'] = USE_COMFYUI_MANAGER
OPTIONS['INSTALL_CUSTOM_NODES_DEPENDENCIES'] = INSTALL_CUSTOM_NODES_DEPENDENCIES

current_dir = !pwd
WORKSPACE = f"{current_dir[0]}/ComfyUI"

# ── Google Drive 連動（USE_GOOGLE_DRIVE フラグで制御）──
if OPTIONS['USE_GOOGLE_DRIVE']:
    !echo "Mounting Google Drive..."
    %cd /
    from google.colab import drive
    drive.mount('/content/drive')
    WORKSPACE = "/content/drive/MyDrive/ComfyUI"
    %cd /content/drive/MyDrive
    print(f"✅ Google Drive モード: 画像は {WORKSPACE}/output に保存されます")
else:
    print(f"✅ ローカルモード: 画像は {WORKSPACE}/output に保存されます")

# ── ComfyUI のクローン / 更新 ──
![ ! -d $WORKSPACE ] && echo -= Initial setup ComfyUI =- && git clone https://github.com/comfyanonymous/ComfyUI
%cd $WORKSPACE

if OPTIONS['UPDATE_COMFY_UI']:
    !echo -= Updating ComfyUI =-
    ![ -f ".ci/nightly/update_windows/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/nightly/update_windows/update_comfyui_and_python_dependencies.bat
    ![ -f ".ci/nightly/windows_base_files/run_nvidia_gpu.bat" ] && chmod 755 .ci/nightly/windows_base_files/run_nvidia_gpu.bat
    ![ -f ".ci/update_windows/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/update_windows/update_comfyui_and_python_dependencies.bat
    ![ -f ".ci/update_windows_cu118/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/update_windows_cu118/update_comfyui_and_python_dependencies.bat
    ![ -f ".ci/update_windows/update.py" ] && chmod 755 .ci/update_windows/update.py
    ![ -f ".ci/update_windows/update_comfyui.bat" ] && chmod 755 .ci/update_windows/update_comfyui.bat
    ![ -f ".ci/update_windows/README_VERY_IMPORTANT.txt" ] && chmod 755 .ci/update_windows/README_VERY_IMPORTANT.txt
    ![ -f ".ci/update_windows/run_cpu.bat" ] && chmod 755 .ci/update_windows/run_cpu.bat
    ![ -f ".ci/update_windows/run_nvidia_gpu.bat" ] && chmod 755 .ci/update_windows/run_nvidia_gpu.bat
    !git pull

# ── 依存パッケージ ──
!echo -= Install dependencies =-
!pip install -q accelerate
!pip install -q einops transformers>=4.28.1 safetensors>=0.4.2 aiohttp pyyaml "Pillow>=10.4.0" scipy tqdm psutil tokenizers>=0.13.3

# 【r2改善①】torch は Colab 既存の cu128 版をそのまま使う（再インストール不要）
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121  ← 削除
print("ℹ️  torch: Colab 既存ビルドを使用 (再インストールをスキップ)")
import torch; print(f"   torch version: {torch.__version__}")

!pip install -q torchsde
!pip install -q kornia>=0.7.1 spandrel soundfile sentencepiece
!pip install -q comfyui-workflow-templates
!pip install -q comfyui-embedded-docs

# 【r2改善②】nodes_math / nodes_glsl エラーを解消する追加パッケージ
!pip install -q simpleeval
!pip install -q pyopengl

# 【r2改善③】fp8/fp4 量子化対応
!pip install -q comfy-kitchen

# ── ComfyUI-Manager ──
if OPTIONS['USE_COMFYUI_MANAGER']:
    %cd custom_nodes
    ![ -f "ComfyUI-Manager/check.sh" ] && chmod 755 ComfyUI-Manager/check.sh
    ![ -f "ComfyUI-Manager/scan.sh" ] && chmod 755 ComfyUI-Manager/scan.sh
    ![ -f "ComfyUI-Manager/node_db/dev/scan.sh" ] && chmod 755 ComfyUI-Manager/node_db/dev/scan.sh
    ![ -f "ComfyUI-Manager/node_db/tutorial/scan.sh" ] && chmod 755 ComfyUI-Manager/node_db/tutorial/scan.sh
    ![ -f "ComfyUI-Manager/scripts/install-comfyui-venv-linux.sh" ] && chmod 755 ComfyUI-Manager/scripts/install-comfyui-venv-linux.sh
    ![ -f "ComfyUI-Manager/scripts/install-comfyui-venv-win.bat" ] && chmod 755 ComfyUI-Manager/scripts/install-comfyui-venv-win.bat
    ![ ! -d ComfyUI-Manager ] && echo -= Initial setup ComfyUI-Manager =- && git clone https://github.com/ltdrdata/ComfyUI-Manager
    %cd ComfyUI-Manager
    !git pull

# ComfyUI-Impact-Pack
![ ! -d $WORKSPACE/custom_nodes/ComfyUI-Impact-Pack ] && git clone https://github.com/ltdrdata/ComfyUI-Impact-Pack.git $WORKSPACE/custom_nodes/ComfyUI-Impact-Pack

%cd $WORKSPACE

if OPTIONS['INSTALL_CUSTOM_NODES_DEPENDENCIES']:
    !echo -= Install custom nodes dependencies =-
    !pip install -q GitPython
    !python custom_nodes/ComfyUI-Manager/cm-cli.py restore-dependencies

# rembg / onnxruntime / insightface
!pip install -q rembg onnxruntime insightface

# av / comfy_aimdo
!pip install -q av comfy_aimdo

print("\n✅ Step 1 完了")

## Step 2: 仮想ディスプレイ設定（GLSLノード有効化）
Colab はヘッドレス環境のため、OpenGL を必要とする `nodes_glsl.py` を使うには xvfb が必要です。

In [ ]:
#@title 【r2改善④】xvfb 仮想ディスプレイのセットアップ（GLSLノード有効化）

!apt-get install -y -q xvfb
import subprocess, os, time

# 既存の :99 プロセスがあれば終了
subprocess.run(["pkill", "-f", "Xvfb :99"], capture_output=True)
time.sleep(0.5)

# 仮想ディスプレイ起動
subprocess.Popen(["Xvfb", ":99", "-screen", "0", "1024x768x24"])
os.environ['DISPLAY'] = ':99'
time.sleep(1)

print("✅ 仮想ディスプレイ :99 を起動しました (DISPLAY=:99)")

## Step 3: 出力フォルダの設定
生成画像を常に Google Drive の  へ保存します。


In [ ]:
#@title 【r2改善⑤】出力先設定（常に Google Drive へ保存）

import os, shutil
from google.colab import drive

# Google Drive マウント
drive.mount("/content/drive", force_remount=False)

# WORKSPACE を再定義（Step1 未実行でも動くよう独立させる）
_workspace = "/content/ComfyUI"

# Drive 側の出力フォルダを作成
drive_output = "/content/drive/MyDrive/ComfyUI_Output"
os.makedirs(drive_output, exist_ok=True)

# ComfyUI の output をシンボリックリンクで差し替え
local_output = f"{_workspace}/output"
if os.path.islink(local_output):
    os.unlink(local_output)
elif os.path.isdir(local_output):
    shutil.rmtree(local_output)

os.symlink(drive_output, local_output)

# 確認
assert os.path.islink(local_output)
assert os.path.exists(local_output)
print("output ->", os.path.realpath(local_output))
print("setup complete: images will be saved to MyDrive/ComfyUI_Output")


## Step 4: モデルのダウンロード
`huggingface_hub` を使ってリトライ付き・整合性チェック付きでダウンロードします。
追加したいモデルはコメントアウトを外してください。

In [ ]:
#@title 【r2改善⑥】Anima モデルのダウンロード（パス修正版）

from huggingface_hub import hf_hub_download
import os, shutil, glob

REPO_ID = "circlestone-labs/Anima"
MODEL_BASE = f"{WORKSPACE}/models"

def download_model(repo_id, hf_filename, local_dir):
    """
    hf_hub_download は filename のサブディレクトリ構造をそのまま再現するため、
    ダウンロード後に目的のフォルダへ移動する。
    """
    os.makedirs(local_dir, exist_ok=True)
    basename = os.path.basename(hf_filename)
    dest = os.path.join(local_dir, basename)
    if os.path.exists(dest):
        print(f"skip (exists): {basename}")
        return
    print(f"downloading: {basename} ...")
    tmp_dir = f"{WORKSPACE}/_hf_tmp"
    downloaded_path = hf_hub_download(
        repo_id=repo_id,
        filename=hf_filename,
        local_dir=tmp_dir,
    )
    shutil.move(downloaded_path, dest)
    shutil.rmtree(tmp_dir, ignore_errors=True)
    print(f"done: {basename} -> {dest}")

# Anima 必須モデル
download_model(REPO_ID,
    "split_files/diffusion_models/anima-preview.safetensors",
    f"{MODEL_BASE}/diffusion_models")

download_model(REPO_ID,
    "split_files/text_encoders/qwen_3_06b_base.safetensors",
    f"{MODEL_BASE}/text_encoders")

download_model(REPO_ID,
    "split_files/vae/qwen_image_vae.safetensors",
    f"{MODEL_BASE}/vae")

# 保存先を確認
print("")
print("saved model files:")
for p in sorted(glob.glob(f"{MODEL_BASE}/**/*.safetensors", recursive=True)):
    print(" ", p)

print("")
print("model download complete")


In [ ]:
#@title オプション: その他のモデル（コメントアウトを外して使用）

# from huggingface_hub import hf_hub_download
# MODEL_BASE = f"{WORKSPACE}/models"

# ── SDXL ──
# hf_hub_download("stabilityai/stable-diffusion-xl-base-1.0",
#     "sd_xl_base_1.0.safetensors", local_dir=f"{MODEL_BASE}/checkpoints", local_dir_use_symlinks=False)
# hf_hub_download("stabilityai/stable-diffusion-xl-refiner-1.0",
#     "sd_xl_refiner_1.0.safetensors", local_dir=f"{MODEL_BASE}/checkpoints", local_dir_use_symlinks=False)

# ── FLUX.1 ──
# hf_hub_download("black-forest-labs/FLUX.1-schnell",
#     "flux1-schnell.safetensors", local_dir=f"{MODEL_BASE}/diffusion_models", local_dir_use_symlinks=False)

# ── VAE (汎用) ──
# hf_hub_download("stabilityai/sd-vae-ft-mse-original",
#     "vae-ft-mse-840000-ema-pruned.safetensors", local_dir=f"{MODEL_BASE}/vae", local_dir_use_symlinks=False)

# ── UpScale ──
# import urllib.request
# urllib.request.urlretrieve(
#     "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
#     f"{MODEL_BASE}/upscale_models/RealESRGAN_x4plus.pth")

print("オプションモデルセルです。必要なものをコメントアウト解除してください。")

## Step 4.5: ControlNet Pose セットアップ（オプション）
キャラクターのポーズを正確に制御したい場合に実行してください。
- `comfyui_controlnet_aux` カスタムノードをインストール（DWPose Estimator が使えるようになります）
- ControlNet OpenPose モデルをダウンロード


In [ ]:
#@title ControlNet Pose セットアップ

import os, shutil
from huggingface_hub import hf_hub_download

_workspace = "/content/ComfyUI"

# ── 1. comfyui_controlnet_aux のインストール ──
print("[1/3] comfyui_controlnet_aux をインストール中...")
cn_aux_dir = f"{_workspace}/custom_nodes/comfyui_controlnet_aux"
if not os.path.exists(cn_aux_dir):
    import subprocess
    subprocess.run(
        ["git", "clone", "https://github.com/Fannovel16/comfyui_controlnet_aux.git", cn_aux_dir],
        check=True
    )
    req = f"{cn_aux_dir}/requirements.txt"
    if os.path.exists(req):
        subprocess.run(["pip", "install", "-q", "-r", req], check=True)
    print("done: comfyui_controlnet_aux")
else:
    print("skip (exists): comfyui_controlnet_aux")

# ── 2. ControlNet OpenPose モデルのダウンロード ──
print("[2/3] ControlNet OpenPose モデルをダウンロード中...")
cn_dir = f"{_workspace}/models/controlnet"
os.makedirs(cn_dir, exist_ok=True)

def download_to(repo_id, filename, dest_dir, save_name=None):
    save_name = save_name or os.path.basename(filename)
    dest = os.path.join(dest_dir, save_name)
    if os.path.exists(dest):
        print(f"skip (exists): {save_name}")
        return dest
    print(f"downloading: {save_name} ...")
    tmp_dir = f"{_workspace}/_dl_tmp"
    path = hf_hub_download(repo_id=repo_id, filename=filename, local_dir=tmp_dir)
    os.makedirs(dest_dir, exist_ok=True)
    shutil.move(path, dest)
    shutil.rmtree(tmp_dir, ignore_errors=True)
    print(f"done: {save_name}")
    return dest

# comfyanonymous 提供の fp16 safetensors 版
download_to(
    "comfyanonymous/ControlNet-v1-1_fp16_safetensors",
    "control_v11p_sd15_openpose_fp16.safetensors",
    cn_dir
)

# ── 3. DWPose 検出モデルのダウンロード ──
# 正しいリポジトリ: yzd-v/DWPose
# 正しいファイル名: dw-ll_ucoco_384.onnx (Pose) / yolox_l.onnx (Det)
print("[3/3] DWPose 検出モデルをダウンロード中...")

# comfyui_controlnet_aux が参照するデフォルトパス
dw_ckpt_dir = f"{cn_aux_dir}/ckpts"
os.makedirs(dw_ckpt_dir, exist_ok=True)

# Pose モデル (134MB)
download_to(
    "yzd-v/DWPose",
    "dw-ll_ucoco_384.onnx",
    dw_ckpt_dir
)

# Det モデル (yolox_l, 217MB)
download_to(
    "yzd-v/DWPose",
    "yolox_l.onnx",
    dw_ckpt_dir
)

# ── 確認 ──
import glob
print("")
print("ControlNet models:")
for p in sorted(glob.glob(f"{cn_dir}/*.safetensors") + glob.glob(f"{cn_dir}/*.pth")):
    print(" ", p)
print("DWPose ckpts:")
for p in sorted(glob.glob(f"{dw_ckpt_dir}/*.onnx")):
    print(" ", p)
print("")
print("ControlNet Pose セットアップ完了")
print("ComfyUI を再起動すると DWPose Estimator ノードが使えるようになります")


### Step 4.6: ポーズ参照画像の準備
正面向きの素体ポーズ画像を自動生成します。手持ちの画像がある場合はスキップしてください。


In [ ]:
#@title ポーズ参照画像を生成（正面T字立ち）

# PIL を使わず cv2 + numpy で生成（Pillow バージョン競合を回避）
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless"], check=True)

import cv2, numpy as np, os, shutil

_workspace = "/content/ComfyUI"
input_dir = f"{_workspace}/input"
os.makedirs(input_dir, exist_ok=True)

W, H = 520, 520
img = np.zeros((H, W, 3), dtype=np.uint8)  # 黒背景

# キーポイント座標（正面T字立ち） x, y
kp = {
    "nose":  (260,  95),
    "neck":  (260, 138),
    "rsho":  (210, 138),
    "relb":  (170, 210),
    "rwri":  (135, 285),
    "lsho":  (310, 138),
    "lelb":  (350, 210),
    "lwri":  (385, 285),
    "rhip":  (225, 290),
    "rknee": (220, 380),
    "rank":  (218, 460),
    "lhip":  (295, 290),
    "lknee": (300, 380),
    "lank":  (302, 460),
}

# cv2 は BGR なので色を逆順で指定
bones = [
    ("nose",  "neck",  (  0, 255, 255)),  # 黄
    ("neck",  "rsho",  (  0,   0, 255)),  # 赤
    ("rsho",  "relb",  (  0, 128, 255)),  # オレンジ
    ("relb",  "rwri",  (  0, 200, 255)),  # 黄オレンジ
    ("neck",  "lsho",  (  0, 255,   0)),  # 緑
    ("lsho",  "lelb",  (128, 200,   0)),  # 黄緑
    ("lelb",  "lwri",  (255, 255,   0)),  # シアン
    ("neck",  "rhip",  (128,   0, 255)),  # ピンク
    ("neck",  "lhip",  (255,   0, 128)),  # 紫
    ("rhip",  "rknee", (200,   0, 200)),  # マゼンタ
    ("rknee", "rank",  (255,   0, 255)),  # 明マゼンタ
    ("lhip",  "lknee", (255, 128,   0)),  # 青
    ("lknee", "lank",  (255,   0,   0)),  # 青
    ("rhip",  "lhip",  (  0, 200, 200)),  # 黄緑
]

for a, b, color in bones:
    cv2.line(img, kp[a], kp[b], color, thickness=4, lineType=cv2.LINE_AA)

# 頭部の円
nx, ny = kp["nose"]
cv2.circle(img, (nx, ny - 5), 33, (0, 255, 255), thickness=3, lineType=cv2.LINE_AA)

# キーポイントの白丸
for name, (x, y) in kp.items():
    cv2.circle(img, (x, y), 5, (255, 255, 255), thickness=-1, lineType=cv2.LINE_AA)

# 保存（cv2 は BGR のまま書き込むので問題なし）
out_path = f"{input_dir}/pose_front_standing.png"
cv2.imwrite(out_path, img)
print(f"saved: {out_path}")

# Drive にもコピー
drive_out = "/content/drive/MyDrive/ComfyUI_Output/pose_front_standing.png"
if os.path.exists("/content/drive/MyDrive"):
    shutil.copy(out_path, drive_out)
    print(f"Drive にもコピー: {drive_out}")

# Colab 上でプレビュー（matplotlib で表示。PIL不要）
import matplotlib.pyplot as plt
plt.figure(figsize=(4, 4))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.tight_layout()
plt.show()
print("ComfyUI の LoadImage でこのファイルを選択してください: pose_front_standing.png")


## Step 5: ComfyUI の起動

**cloudflared（推奨）** か **localtunnel** か **Colab iframe** の3種類から選んで実行してください。

> ControlNet Pose を使う場合は Step 4.5 → Step 4.6 を先に実行してから起動してください。


In [ ]:
#@title 起動方法 A: cloudflared（推奨）

!wget -q -P ~ https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i ~/cloudflared-linux-amd64.deb

import subprocess
import threading
import time
import socket

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()
    print("\nComfyUI の起動完了。cloudflared でトンネルを開きます...\n")
    p = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for line in p.stderr:
        l = line.decode()
        if "trycloudflare.com " in l:
            print("🌐 ComfyUI アクセス URL:", l[l.find("http"):], end='')

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

%cd /content/ComfyUI

# GPU が使えない場合は --cpu フラグを追加してください
# !python main.py --cpu --dont-print-server
!python main.py --dont-print-server

In [ ]:
#@title 起動方法 B: localtunnel（cloudflared が使えない場合）

!npm install -g localtunnel

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()
    print("\nComfyUI の起動完了。localtunnel でトンネルを開きます...\n")
    endpoint_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
    print("🔑 localtunnel パスワード / エンドポイント IP:", endpoint_ip)
    p = subprocess.Popen(["lt", "--port", str(port)], stdout=subprocess.PIPE)
    for line in p.stdout:
        print(line.decode(), end='')

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

%cd /content/ComfyUI
!python main.py --dont-print-server

In [ ]:
#@title 起動方法 C: Colab iframe（WebSocket 非対応のため機能制限あり）

import threading
import time
import socket

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()
    from google.colab import output
    output.serve_kernel_port_as_iframe(port, height=1024)
    print("別ウィンドウで開く場合はこちら:")
    output.serve_kernel_port_as_window(port)

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

%cd /content/ComfyUI
!python main.py --dont-print-server